In [ ]:
!rm -rf Classical-Chinese-Summarization
!git clone https://github.com/ctaiyi15/Classical-Chinese-Summarization.git
!ls Classical-Chinese-Summarization/data

Cloning into 'Classical-Chinese-Summarization'...
remote: Enumerating objects: 7510, done.
remote: Counting objects: 100% (2126/2126), done.
remote: Compressing objects: 100% (2115/2115), done.
remote: Total 7510 (delta 427), reused 1698 (delta 9), pack-reused 5384 (from 1)
Receiving objects: 100% (7510/7510), 66.14 MiB | 8.34 MiB/s, done.
Resolving deltas: 100% (1249/1249), done.
Updating files: 100% (3737/3737), done.
dataset.json  processed  raw  segmented  segmented_v2


In [ ]:
!ls Classical-Chinese-Summarization/data/processed
!pip install -U openai nest_asyncio
import asyncio
import json
import re
from pathlib import Path

import nest_asyncio
from openai import AsyncOpenAI
from google.colab import userdata
from tqdm.notebook import tqdm

input_root = Path(
    "Classical-Chinese-Summarization/data/processed/translated"
)

summary_root = Path(
    "Classical-Chinese-Summarization/data/processed/summary_clean"
)

evaluation  sample  summary  summary_clean  translated	translated_back


In [ ]:
def safe_json_loads(content, debug_info=None):

    try:
        return json.loads(content)

    except json.JSONDecodeError as original_error:

        raw_content = content

        content = content.strip()

        content = re.sub(r"^```json", "", content)
        content = re.sub(r"^```", "", content)
        content = re.sub(r"```$", "", content)

        match = re.search(
            r"\{.*\}",
            content,
            re.DOTALL
        )

        if match:
            try:
                return json.loads(match.group())

            except json.JSONDecodeError as extracted_error:

                print("\n" + "=" * 80)
                print("JSON ERROR AFTER EXTRACTING JSON-LIKE OBJECT")
                print("=" * 80)

                if debug_info is not None:
                    print("DEBUG INFO:")
                    print(
                        json.dumps(
                            debug_info,
                            ensure_ascii=False,
                            indent=2
                        )
                    )

                print("\nRAW MODEL OUTPUT:")
                print(raw_content)
                print("=" * 80)

                raise extracted_error

        print("\n" + "=" * 80)
        print("JSON ERROR: NO VALID JSON OBJECT FOUND")
        print("=" * 80)

        if debug_info is not None:
            print("DEBUG INFO:")
            print(
                json.dumps(
                    debug_info,
                    ensure_ascii=False,
                    indent=2
                )
            )

        print("\nRAW MODEL OUTPUT:")
        print(raw_content)
        print("=" * 80)

        raise original_error

nest_asyncio.apply()
client = AsyncOpenAI(
    api_key=userdata.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
MAX_CONCURRENT_REQUESTS = 10

SEM = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
async def guarded_completion(messages, temperature=0, max_tokens=10000):

    async with SEM:

        response = await client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            temperature=temperature,
            response_format={"type": "json_object"},
            max_tokens=max_tokens
        )

    return response

def deduplicate_claims(claims):
    seen = set()
    unique = []

    for claim in claims:
        normalized = re.sub(
            r"\s+",
            " ",
            claim.strip().lower()
        )

        if normalized not in seen:
            seen.add(normalized)
            unique.append(claim.strip())

    return unique

async def extract_claims(summary, debug_info=None):

    prompt = f"""
You are a precise information extraction system.

Task:
Break the summary into atomic factual claims.

Rules:
- Each claim must contain ONE fact only
- No merging multiple facts
- Do NOT add new information
- Keep claims short and concrete

Return JSON ONLY:
{{
  "claims": ["claim 1", "claim 2"]
}}

SUMMARY:
{summary}
"""

    response = await guarded_completion(
        [{"role": "user", "content": prompt}]
    )

    content = response.choices[0].message.content

    # return safe_json_loads(content)["claims"]

    return safe_json_loads(
        content,
        debug_info=debug_info
    )["claims"]

async def extract_claims_chunked(summary, debug_info=None):

    chunks = chunk_text_by_sentences(
        summary,
        max_words=500,
        overlap_sentences=1
    )

    all_claims = []

    print("\n" + "=" * 88)
    print(f"Sentence-claimed Chunk Extraction [{summary[:6]}]:", end=" ")
    print("Summary words:", len(summary.split()), end="; ")
    print("Number of chunks:", len(chunks))
    print("=" * 88)

    for i, chunk in enumerate(chunks, start=1):

        chunk_debug_info = {
            **(debug_info or {}),
            "stage": "extract_claims",
            "chunk_index": i,
            "total_chunks": len(chunks),
            "chunk_words": len(chunk.split()),
            "chunk_preview": chunk[:500]
        }

        chunk_claims = await extract_claims(
            chunk,
            debug_info=chunk_debug_info
        )

        all_claims.extend(chunk_claims)

    all_claims = deduplicate_claims(all_claims)

    print(f"\nTotal unique claims [{summary[:6]}]:", len(all_claims))

    return all_claims

async def retrieve_relevant_parts(claim, source_text, debug_info=None):

    prompt = f"""
You are an information retrieval system.

Task:
Extract ONLY the parts of the source text that are relevant to verifying the claim.

Rules:
- Copy exact spans from the source text
- Do NOT paraphrase
- If nothing is relevant, return empty list
- Keep output minimal (max 5 excerpts)

Return JSON ONLY:
{{
  "evidence": ["excerpt 1", "excerpt 2"]
}}

SOURCE TEXT:
{source_text}

CLAIM:
{claim}
"""

    response = await guarded_completion(
        [{"role": "user", "content": prompt}]
    )

    content = response.choices[0].message.content

    return safe_json_loads(
        content,
        debug_info=debug_info
    )["evidence"]

async def verify_claim(claim, evidence_list, debug_info=None):

    context = "\n\n".join(evidence_list)

    prompt = f"""
You are a strict fact-checking system.

Task:
Determine whether the claim is supported by the evidence.

Rules:
- ONLY use the provided evidence.
- Judge factual meaning, not exact wording.
- Paraphrases are supported if the meaning is the same.
- Minor tense/aspect differences are acceptable in historical summaries if they do not change the event or fact.
- If the evidence supports only part of the claim, label it "partially_supported".
- If the evidence clearly indicates an action was underway but does not clearly show completion, label it "partially_supported".
- Do NOT label plans, intentions, attempts, preparations, or possibilities as fully "supported" unless the evidence implies completion.
- If evidence is insufficient, label it "unsupported".
- Be strict about factual content, but not about harmless wording or tense differences.

Return JSON ONLY:
{{
  "label": "supported"
}}

Allowed labels:
- "supported"
- "partially_supported"
- "unsupported"

EVIDENCE:
{context}

CLAIM:
{claim}
"""

    response = await guarded_completion(
        [{"role": "user", "content": prompt}]
    )

    content = response.choices[0].message.content

    return safe_json_loads(
        content,
        debug_info=debug_info
    )

async def process_claim(claim, source_text, debug_info=None):

    evidence = await retrieve_relevant_parts(
        claim,
        source_text,
        debug_info={
            **(debug_info or {}),
            "stage": "retrieve_relevant_parts",
            "claim": claim
        }
    )

    verdict = await verify_claim(
        claim,
        evidence,
        debug_info={
            **(debug_info or {}),
            "stage": "verify_claim",
            "claim": claim,
            "evidence": evidence
        }
    )

    return {
        "claim": claim,
        "evidence": evidence,
        "label": verdict["label"]
    }

async def evaluate_faithfulness(summary, source_text, debug_info=None):

    claims = await extract_claims_chunked(
        summary,
        debug_info=debug_info
    )

    tasks = [
        process_claim(
            claim,
            source_text,
            debug_info={
                **(debug_info or {}),
                "claim_index": idx,
                "total_claims": len(claims)
            }
        )
        for idx, claim in enumerate(claims, start=1)
    ]

    results = await asyncio.gather(*tasks)

    supported = sum(
        r["label"] == "supported"
        for r in results
    )

    partially_supported = sum(
        r["label"] == "partially_supported"
        for r in results
    )

    unsupported = sum(
        r["label"] == "unsupported"
        for r in results
    )

    total = len(results)

    score = (
        supported + 0.5 * partially_supported
    ) / total if total > 0 else 0

    return {
        "faithfulness_score": score,
        "total_claims": total,
        "supported_claims": supported,
        "partially_supported_claims": partially_supported,
        "unsupported_claims": unsupported,
        "details": results
    }



def build_pairs():

    pairs = []

    source_files = list(input_root.rglob("*.txt"))

    for src_path in source_files:

        rel = src_path.relative_to(input_root)

        summary_path = (
            summary_root / rel
        ).with_suffix(".summary.txt")

        if summary_path.exists():
            pairs.append((src_path, summary_path))

    return pairs

def read_file(path):

    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

async def process_file(src_path, sum_path):

    source_text = read_file(src_path)

    summary_text = read_file(sum_path)

    debug_info = {
        "source_file": str(src_path),
        "summary_file": str(sum_path)
    }

    result = await evaluate_faithfulness(
        summary_text,
        source_text,
        debug_info=debug_info
    )

    return {
        "source_file": str(src_path),
        "summary_file": str(sum_path),
        **result
    }

async def process_file_with_paths(src, summ):
    try:
        result = await process_file(src, summ)
        return {
            "ok": True,
            "source_file": str(src),
            "summary_file": str(summ),
            "result": result
        }
    except Exception as e:
        return {
            "ok": False,
            "source_file": str(src),
            "summary_file": str(summ),
            "error_type": type(e).__name__,
            "error": str(e),
            "repr": repr(e)
        }

LIVE_SAVE_PATH = "faithfulness_live_results.json"
FAILED_SAVE_PATH = "faithfulness_failed_files.json"

async def run_first_x_live_saved(x):
# async def run_all_live_saved():

    pairs = build_pairs()[:x]

    print(f"Running on {len(pairs)} file pairs")

    results = []
    errors = []

    tasks = [
        process_file_with_paths(src, summ)
        for src, summ in pairs
    ]

    completed = 0
    failed = 0

    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks)):

        item = await coro

        if item["ok"]:

            result = item["result"]
            results.append(result)
            completed += 1

            score = result["faithfulness_score"]

            avg = sum(
                r["faithfulness_score"]
                for r in results
            ) / len(results)

            print("\n" + "=" * 60)
            print(f"SUCCESS [{completed}/{len(tasks)}]")
            print("Source file:", item["source_file"])
            print("Summary file:", item["summary_file"])
            print("Score:", score)
            print("Running avg:", avg)

            display_result = {
                k: v
                for k, v in result.items()
                if k != "details"
            }

            print(
                json.dumps(
                    display_result,
                    ensure_ascii=False,
                    indent=2
                )
            )

            with open(
                LIVE_SAVE_PATH,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    results,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

        else:

            failed += 1
            errors.append(item)

            print("\n" + "=" * 60)
            print(f"FAILED [{failed}/{len(tasks)}]")
            print("Source file:", item["source_file"])
            print("Summary file:", item["summary_file"])
            print("ERROR:", item["repr"])

            with open(
                FAILED_SAVE_PATH,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    errors,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

    print("\nDONE")
    print("Successful files:", len(results))
    print("Failed files:", len(errors))

    if results:
        final_avg = sum(
            r["faithfulness_score"]
            for r in results
        ) / len(results)

        print("Final avg:", final_avg)
    else:
        print("No successful results. Final avg cannot be computed.")

    if errors:
        print("Failed file list saved to:", FAILED_SAVE_PATH)

    return results


async def run_first_summary_debug():

    pairs = build_pairs()[:1]

    print(f"Running on {len(pairs)} file pair")

    if not pairs:
        print("No file pairs found")
        return None

    src_path, sum_path = pairs[0]

    print("Source file:", src_path)
    print("Summary file:", sum_path)

    result = await process_file(src_path, sum_path)

    display_result = {
        k: v
        for k, v in result.items()
        if k != "details"
    }

    print(
        json.dumps(
            display_result,
            ensure_ascii=False,
            indent=2
        )
    )

    return result

import re

def split_into_sentences(text):
    """
    Simple sentence splitter for English translated summaries.
    Keeps sentence-final punctuation attached.
    """
    text = re.sub(r"\s+", " ", text.strip())

    if not text:
        return []

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    return [
        s.strip()
        for s in sentences
        if s.strip()
    ]

def chunk_text_by_sentences(text, max_words=1000, overlap_sentences=1):
    sentences = split_into_sentences(text)

    chunks = []
    current = []
    current_words = 0

    for sentence in sentences:
        sentence_words = len(sentence.split())

        # if adding this sentence would exceed max_words, close the current chunk first
        if current and current_words + sentence_words > max_words:
            chunks.append(" ".join(current))

            # keep a small sentence overlap for continuity
            if overlap_sentences > 0:
                current = current[-overlap_sentences:]
                current_words = sum(
                    len(s.split())
                    for s in current
                )
            else:
                current = []
                current_words = 0

        current.append(sentence)
        current_words += sentence_words

    if current:
        chunks.append(" ".join(current))

    return chunks


In [ ]:
results = await run_first_x_live_saved(1000)

Running on 409 file pairs


  0%|          | 0/409 [00:00<?, ?it/s]

流式输出内容被截断，只能显示最后 5000 行内容。
Source file: Classical-Chinese-Summarization/data/processed/translated/史记/七十列传/张丞相列传/target.txt
Summary file: Classical-Chinese-Summarization/data/processed/summary_clean/史记/七十列传/张丞相列传/target.summary.txt
Score: 0.6730769230769231
Running avg: 0.7422977243858321
{
  "source_file": "Classical-Chinese-Summarization/data/processed/translated/史记/七十列传/张丞相列传/target.txt",
  "summary_file": "Classical-Chinese-Summarization/data/processed/summary_clean/史记/七十列传/张丞相列传/target.summary.txt",
  "faithfulness_score": 0.6730769230769231,
  "total_claims": 78,
  "supported_claims": 52,
  "partially_supported_claims": 1,
  "unsupported_claims": 25
}

Total unique claims [The Ha]: 167

SUCCESS [103/409]
Source file: Classical-Chinese-Summarization/data/processed/translated/史记/七十列传/孙子吴起列传/target.txt
Summary file: Classical-Chinese-Summarization/data/processed/summary_clean/史记/七十列传/孙子吴起列传/target.summary.txt
Score: 0.7803030303030303
Running avg: 0.7426667079384262
{
  "source_file"

In [ ]:
from google.colab import files

files.download("faithfulness_live_results.json")
files.download("faithfulness_failed_files.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: faithfulness_failed_files.json